# 03 - Neighborhood "feel" features (quiet, green, access, form, centrality)

Replaces the shop-mix similarity from notebook 02 with features that match what I actually care about:
quiet streets, green space, access to a library/park/groceries at bike and drive scale, moderate density,
and moderate distance to downtown. **No income and no boutique/shop-mix features.** Price will be a budget filter later, not a similarity feature.

Inputs: the parquet cache from notebook 02 (POIs, boundaries) plus extracts from the same Geofabrik PBF (road classes, street form from the driving network, parks, trails, libraries, buildings). `data/labels.csv` holds the labeled places.

Test before trusting anything (write the answer down first): does the home-area anchor's nearest-Houston list include West U and Museum District? Do Midtown and Richmond-like places look different?

Similarity here is **weighted Euclidean distance on z-scored features**, not cosine: absolute level matters (a quiet street is not the same as "a bit less busy than average").

In [ ]:
# Kaggle/Colab: uncomment. Locally: pip install in your venv instead.
# %pip install -q geopandas pyarrow pyrosm "shapely>=2" scipy

In [ ]:
import os, warnings
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, shapely
from shapely import STRtree
from scipy.spatial import cKDTree
from pyproj import Transformer
from pyrosm import OSM
warnings.filterwarnings("ignore")

CACHE = Path("/kaggle/working/cache") if os.path.exists("/kaggle") else Path("../data/interim")
PBF_DIR = CACHE / "pbf"
STATE = {"houston": "texas", "austin": "texas", "seattle": "washington", "atlanta": "georgia"}
CITIES = ["houston", "atlanta", "austin", "seattle"]   # each non-Houston city needs notebook 02 run first (boundary + POIs + streets from its state PBF)
# Approximate downtown points (lat, lon) - VERIFY ON A MAP
DOWNTOWN = {"houston": (29.7604, -95.3698), "austin": (30.2672, -97.7431),
            "seattle": (47.6062, -122.3321), "atlanta": (33.7490, -84.3880)}
# Census (ACS) block-group features: housing type by tenure (condos vs apartment complexes vs townhouses vs houses),
# households with children / 65+, owner share, plus median rent and home value as FILTERS (not similarity).
# Needs a free Census API key once (https://api.census.gov/data/key_signup.html); the result is cached and goes into the dataset.
# USE_ACS = False falls back to the OSM-only housing guess (mf_share, small_home_share).
USE_ACS = True
ACS_YEAR = 2023            # ACS 5-year estimates 2019-2023, with the matching TIGER block-group shapes
STATE_FIPS = {"texas": "48", "georgia": "13", "washington": "53"}
CENSUS_KEY = os.environ.get("CENSUS_API_KEY")
if CENSUS_KEY is None and os.path.exists("/kaggle"):
    try:
        from kaggle_secrets import UserSecretsClient
        CENSUS_KEY = UserSecretsClient().get_secret("CENSUS_API_KEY")
    except Exception:
        pass
CENSUS_KEY = CENSUS_KEY.strip().strip("'\"") if CENSUS_KEY else CENSUS_KEY   # pasted keys often carry a space/newline/quotes
SPACING_M, RADIUS_M, NMS_M = 400, 800, 1600
BIG_PARK_HA = 10         # "large park" threshold (Hermann Park is far above this)
MIN_POI_800 = 10          # drop empty land: cell needs this many POIs of any kind within 800 m

def city_crs(bbox):
    lon = (bbox[0] + bbox[2]) / 2
    return f"EPSG:{32600 + int((lon + 180) // 6) + 1}"   # must match notebook 02

def latest(pattern):
    fs = list(CACHE.glob(pattern))
    if not fs: raise FileNotFoundError(f"{pattern} not in {CACHE}; run notebook 02 first")
    if len(fs) == 1: return fs[0]
    # Several cache keys can coexist (old Houston-only vs Houston+enclaves). mtimes don't survive Kaggle
    # datasets reliably, so prefer the file with the most rows: the union area is a superset of the old one.
    import pyarrow.parquet as pq
    def nrows(p):
        try: return pq.ParquetFile(p).metadata.num_rows
        except Exception: return -1
    fs.sort(key=lambda p: (nrows(p), p.stat().st_mtime))
    print(f"{pattern}: {len(fs)} candidates, using {fs[-1].name} ({nrows(fs[-1])} rows); others: " + ", ".join(f"{p.name} ({nrows(p)})" for p in fs[:-1]))
    return fs[-1]

def boundary_file(city):
    # notebook 02 writes a circle-around-downtown boundary (preferred), or a hash-named union, or the plain city file
    by_time = lambda pat: sorted(CACHE.glob(pat), key=lambda p: p.stat().st_mtime)
    fs = by_time(f"{city}_boundary_ring_*.geojson") or by_time(f"{city}_boundary_circle_*.geojson") or by_time(f"{city}_boundary_*.geojson") or [CACHE / f"{city}_boundary.geojson"]
    return fs[-1]

def pbf_path(city):
    name = f"{STATE[city]}-latest.osm.pbf"
    local = PBF_DIR / name
    if local.exists(): return local
    found = next(Path("/kaggle/input").glob(f"**/{name}"), None)   # a PBF saved in an attached dataset
    return found or local

# labels: ../data/labels.csv locally; on Kaggle upload it to /kaggle/working or attach it as a dataset
cands = [Path("../data/labels.csv"), Path("/kaggle/working/labels.csv")] + list(Path("/kaggle/input").glob("**/labels.csv"))
LABELS = next((p for p in cands if p.exists()), None)
assert LABELS is not None, "labels.csv not found"
labels = pd.read_csv(LABELS).dropna(subset=["lat", "lon"]).reset_index(drop=True)
print(labels[["name", "city", "label"]].to_string(index=False))

## 0b. Setup on Kaggle: copy the saved cache out of the read-only dataset, download the PBF only if needed

In [ ]:
import shutil, subprocess, urllib.request
CACHE.mkdir(parents=True, exist_ok=True)

# 1) Attached datasets are read-only under /kaggle/input. Copy the cached parquet/geojson into the working cache.
for src in Path("/kaggle/input").glob("**/*"):
    if src.suffix in (".parquet", ".geojson") and not (CACHE / src.name).exists():
        shutil.copy2(src, CACHE / src.name); print("copied", src.name)

# 2) Road classes and park polygons need the PBF. Skip the big download if those extracts are already cached.
need_pbf = any(not (CACHE / f"{c}_roads_class.parquet").exists() or not (CACHE / f"{c}_major_pts.parquet").exists()
               or not (CACHE / f"{c}_parks.parquet").exists() or not (CACHE / f"{c}_trail_ways.parquet").exists()
               or not (CACHE / f"{c}_libraries.parquet").exists() or not (CACHE / f"{c}_streets_drive.parquet").exists()
               or not (CACHE / f"{c}_buildings.parquet").exists() for c in CITIES)
if need_pbf:
    PBF_DIR.mkdir(exist_ok=True)
    for s in sorted({STATE[c] for c in CITIES}):
        f = PBF_DIR / f"{s}-latest.osm.pbf"
        if f.exists() or next(Path("/kaggle/input").glob(f"**/{s}-latest.osm.pbf"), None): continue
        part = f.with_name(f.name + ".part"); url = f"https://download.geofabrik.de/north-america/us/{s}-latest.osm.pbf"
        print("downloading", url)
        if part.exists() and part.stat().st_size == 0: part.unlink()   # an empty leftover makes `wget -c` fail
        try:
            if not shutil.which("wget"): raise FileNotFoundError("wget")
            subprocess.run(["wget", "-nv", "-c", "-O", str(part), url], check=True)   # -nv still prints the server's error
        except (subprocess.CalledProcessError, FileNotFoundError) as e:
            print("wget failed (", e, "), retrying with urllib")
            part.unlink(missing_ok=True); urllib.request.urlretrieve(url, part)
        part.rename(f)
print("cache files:", len(list(CACHE.glob("*.parquet"))), "parquet,", len(list(CACHE.glob("*.geojson"))), "geojson")

## 1. Extra extracts from the PBF (cached): road classes and park polygons

In [ ]:
MAJOR = {"motorway", "motorway_link", "trunk", "trunk_link", "primary", "primary_link", "secondary", "secondary_link"}
FREEWAY = {"motorway", "motorway_link", "trunk", "trunk_link"}

def load_roads(city, bbox, crs):
    fr, fm = CACHE / f"{city}_roads_class.parquet", CACHE / f"{city}_major_pts.parquet"
    if fr.exists() and fm.exists():
        return pd.read_parquet(fr), pd.read_parquet(fm)
    ed = OSM(str(pbf_path(city)), bounding_box=list(bbox)).get_network(network_type="driving").to_crs(crs)
    ed = ed[ed.geometry.notna()].reset_index(drop=True)
    hw = ed["highway"].astype(str)
    mid = ed.geometry.interpolate(0.5, normalized=True)
    roads = pd.DataFrame({"x": mid.x.values, "y": mid.y.values, "len": ed.geometry.length.values,
                          "major": hw.isin(MAJOR).values, "freeway": hw.isin(FREEWAY).values})
    rows = []   # points every ~50 m along major roads, for nearest-distance queries
    for geom, fw in zip(ed.geometry[roads["major"]], roads["freeway"][roads["major"]]):
        for t in np.linspace(0, 1, max(int(geom.length // 50), 1) + 1):
            p = geom.interpolate(t, normalized=True); rows.append((p.x, p.y, bool(fw)))
    pts = pd.DataFrame(rows, columns=["x", "y", "freeway"])
    roads.to_parquet(fr); pts.to_parquet(fm)
    return roads, pts

# Street form from the DRIVING network (the walking network counts sidewalks/crossings as edges, so its intersection
# density and block length mostly measured how well sidewalks are mapped, e.g. West U has none mapped).
# Kept: streets people live on or drive through. Dropped: freeways, ramps/links, service roads (driveways, parking aisles), tracks.
KEEP_HW = {"primary", "secondary", "tertiary", "unclassified", "residential", "living_street"}
LOCAL_HW = {"residential", "living_street"}

def load_street_form(city, bbox, crs):
    f = CACHE / f"{city}_streets_drive.parquet"
    if f.exists():
        return pd.read_parquet(f)
    nd, ed = OSM(str(pbf_path(city)), bounding_box=list(bbox)).get_network(network_type="driving", nodes=True)
    ed = ed[ed["highway"].astype(str).isin(KEEP_HW) & ed.geometry.notna()].to_crs(crs)
    nd = nd.to_crs(crs)
    ends = pd.concat([ed["u"], ed["v"]]).value_counts()          # edge ends touching each node (edges are split at junctions)
    deg = nd["id"].map(ends).fillna(0).astype(int)
    n = pd.DataFrame({"kind": "node", "x": nd.geometry.x.values, "y": nd.geometry.y.values, "deg": deg.values,
                      "len": np.nan, "local": False})
    mid = ed.geometry.interpolate(0.5, normalized=True)
    e = pd.DataFrame({"kind": "edge", "x": mid.x.values, "y": mid.y.values, "deg": 0, "len": ed.geometry.length.values,
                      "local": ed["highway"].astype(str).isin(LOCAL_HW).values})
    out = pd.concat([n[n["deg"] > 0], e], ignore_index=True)
    out.to_parquet(f)
    return out

# --- Census ACS block groups: tenure by units in structure (B25032), households with kids (B11005) / 65+ (B11007),
#     median gross rent (B25064), median home value (B25077). 32 variables, so one API call per county.
ACS_VARS = ([f"B25032_{i:03d}E" for i in range(1, 24)] + ["B25003_001E", "B25003_002E", "B11005_001E", "B11005_002E",
            "B11007_001E", "B11007_002E", "B25064_001E", "B25077_001E"])
ACS_NUM = ["occ", "owner", "renter", "big_owner", "big_renter", "att", "det", "hh", "kids", "hh65", "seniors"]
# feature = (numerator, denominator); shrunk toward the city total when little housing is nearby
SHARES = {"rental_complex_share": ("big_renter", "occ"),   # renter-occupied, 5+ unit buildings = apartment complexes
          "condo_share": ("big_owner", "occ"),             # owner-occupied, 5+ unit buildings = mostly condos
          "attached_share": ("att", "occ"),                # 1-unit attached + 2-4 units = townhouses, duplexes, fourplexes
          "detached_share": ("det", "occ"),                # single-family houses
          "owner_share": ("owner", "occ"),
          "kids_share": ("kids", "hh"), "senior_share": ("seniors", "hh65")}
ACS_K = 50

def _fetch(url, dest=None):
    import urllib.request, shutil
    req = urllib.request.Request(url, headers={"User-Agent": "neighborhood-similarity-research/0.1"})
    with urllib.request.urlopen(req, timeout=300) as r:
        if dest is None: return r.read()
        with open(dest, "wb") as out: shutil.copyfileobj(r, out)

def _acs_county(fips, cty):
    import urllib.parse, json
    err = None
    for geo in (f"state:{fips} county:{cty}", f"state:{fips} county:{cty} tract:*"):   # some API versions want the tract level spelled out
        q = urllib.parse.urlencode({"get": ",".join(ACS_VARS), "for": "block group:*", "in": geo, "key": CENSUS_KEY},
                                   quote_via=urllib.parse.quote, safe=":*,")
        try:
            data = json.loads(_fetch(f"https://api.census.gov/data/{ACS_YEAR}/acs/acs5?{q}"))
            return pd.DataFrame(data[1:], columns=data[0])
        except Exception as e:
            body = ""
            if hasattr(e, "read"):   # HTTPError: the response text says WHY (invalid key, bad geography, ...)
                try: body = e.read().decode("utf-8", "replace")[:200].replace(CENSUS_KEY, "***")
                except Exception: pass
            err = type(e).__name__ + ": " + str(e)[:100] + (" | " + body if body else "")
    raise RuntimeError(f"Census API failed for state {fips} county {cty} ({err})")

def load_acs(city, poly4326, crs):
    f = CACHE / f"{city}_acs_bg_{ACS_YEAR}.parquet"
    if f.exists():
        return gpd.read_parquet(f)
    assert CENSUS_KEY, ("No Census API key. Get a free one at https://api.census.gov/data/key_signup.html, save it as a Kaggle secret named "
                        "CENSUS_API_KEY (or the env var CENSUS_API_KEY). It is only needed once: the result is cached and saved in the dataset. "
                        "Or set USE_ACS = False to use the OSM-only housing guess.")
    fips = STATE_FIPS[STATE[city]]
    z = CACHE / f"tiger_bg_{ACS_YEAR}_{fips}.zip"
    if not z.exists():
        print("downloading TIGER block groups for state", fips)
        _fetch(f"https://www2.census.gov/geo/tiger/TIGER{ACS_YEAR}/BG/tl_{ACS_YEAR}_{fips}_bg.zip", z)
    bg = gpd.read_file(f"zip://{z}").to_crs(4326)
    bg = bg[bg.intersects(poly4326)][["GEOID", "COUNTYFP", "geometry"]]
    acs = pd.concat([_acs_county(fips, cty) for cty in sorted(bg["COUNTYFP"].unique())], ignore_index=True)
    acs["GEOID"] = acs["state"] + acs["county"] + acs["tract"] + acs["block group"]
    v = acs[ACS_VARS].apply(pd.to_numeric, errors="coerce"); v = v.where(v >= 0)   # negative codes mean "not available"
    g = lambda *ks: sum(v[k].fillna(0) for k in ks)
    tab = pd.DataFrame({
        "GEOID": acs["GEOID"], "occ": g("B25032_001E"), "owner": g("B25032_002E"), "renter": g("B25032_013E"),
        "big_owner": g(*[f"B25032_{i:03d}E" for i in range(7, 11)]), "big_renter": g(*[f"B25032_{i:03d}E" for i in range(18, 22)]),
        "att": g("B25032_004E", "B25032_005E", "B25032_006E", "B25032_015E", "B25032_016E", "B25032_017E"),
        "det": g("B25032_003E", "B25032_014E"),
        "hh": g("B11005_001E"), "kids": g("B11005_002E"), "hh65": g("B11007_001E"), "seniors": g("B11007_002E"),
        "rent_med": v["B25064_001E"], "val_med": v["B25077_001E"]})
    out = bg.merge(tab, on="GEOID", how="inner").drop(columns="COUNTYFP").to_crs(crs)
    out = out[out.geometry.notna() & ~out.geometry.is_empty].reset_index(drop=True)
    out.to_parquet(f)
    return out

def load_parks(city, bbox, crs):
    f = CACHE / f"{city}_parks.parquet"
    if f.exists():
        return gpd.read_parquet(f)
    g = OSM(str(pbf_path(city)), bounding_box=list(bbox)).get_data_by_custom_criteria(
        custom_filter={"leisure": ["park", "garden", "nature_reserve"]}, filter_type="keep",
        keep_nodes=False, keep_ways=True, keep_relations=True)
    g = g[g.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].to_crs(crs)
    g = g[["geometry"]].reset_index(drop=True)
    g["area_ha"] = g.geometry.area / 1e4
    g.to_parquet(f)
    return g

import re
# Greenway / exercise-path filter. A path (cycleway, path or footway, minus sidewalks/crossings) counts if ANY of:
#  (a) its name looks like a greenway or trail, (b) it is a standalone cycleway of at least MIN_CYCLEWAY_M,
#  (c) it lies inside a big park (>= BIG_PARK_HA). Everything is applied on each run, so tune and rerun without re-extracting.
GREENWAY_RE = re.compile(r"greenway|trail|bayou|creek|hike|bike|loop|river|lake|rail", re.I)
MIN_CYCLEWAY_M = 300

def load_trails(city, bbox, crs, big_parks):
    f = CACHE / f"{city}_trail_ways.parquet"
    if f.exists():
        ways = gpd.read_parquet(f)
    else:
        g = OSM(str(pbf_path(city)), bounding_box=list(bbox)).get_data_by_custom_criteria(
            custom_filter={"highway": ["cycleway", "path", "footway"]}, filter_type="keep",
            keep_nodes=False, keep_ways=True, keep_relations=False)
        g = g[g.geometry.geom_type.isin(["LineString", "MultiLineString"])].to_crs(crs)
        if "footway" in g:
            g = g[~g["footway"].isin(["sidewalk", "crossing", "access_aisle", "traffic_island", "link"])]
        if "access" in g:
            g = g[~g["access"].isin(["private", "no"])]
        col = lambda k: g[k].fillna("").astype(str).values if k in g else np.full(len(g), "", dtype=object)
        ways = gpd.GeoDataFrame({"name": col("name"), "highway": col("highway"), "length_m": g.geometry.length.values},
                                geometry=g.geometry.values, crs=crs)
        ways.to_parquet(f)
    rp = ways.geometry.representative_point()
    in_park = (shapely.contains_xy(shapely.union_all(big_parks.geometry.values), rp.x.values, rp.y.values)
               if len(big_parks) else np.zeros(len(ways), dtype=bool))
    by_name = ways["name"].str.contains(GREENWAY_RE)
    by_cycleway = (ways["highway"] == "cycleway") & (ways["length_m"] >= MIN_CYCLEWAY_M)
    keep = (by_name | by_cycleway | in_park)
    print(f"{city}: trail ways kept {int(keep.sum())} of {len(ways)} (name {int(by_name.sum())}, long cycleway {int(by_cycleway.sum())}, in big park {int(in_park.sum())})")
    print("  longest kept named trails (m):", ways[keep & (ways["name"] != "")].groupby("name")["length_m"].sum().nlargest(12).round().to_dict())
    rows = []   # points every ~50 m along each kept line
    for geom in ways.geometry[keep]:
        for ln in (geom.geoms if geom.geom_type == "MultiLineString" else [geom]):
            for s in np.arange(0, max(ln.length, 1), 50):
                p = ln.interpolate(s); rows.append((p.x, p.y))
    return pd.DataFrame(rows, columns=["x", "y"])

import re
# OSM has no consistent "public library" tag, so drop libraries that look like school/university/private ones.
NONPUBLIC_RE = re.compile(r"\b(?:school|elementary|middle|high school|academy|college|university|campus|isd|seminary|montessori|church|prep)\b", re.I)
# A name/operator that clearly says "public library / branch / neighborhood library" overrides the school/university words
# (e.g. "Fort Bend County Libraries - University Branch").
PUBLIC_RE = re.compile(r"\b(?:public librar|neighborhood librar|branch|regional librar|county librar|express\b)", re.I)
# Manual overrides by exact name (applied on every run). Edit after reading the kept/dropped lists below.
LIB_KEEP = {"West University Library"}
LIB_DROP = {"M.D. Anderson Library", "HCC WestLoop Library", "HAM-TMC Library", "Woodson Research Center",
            "Lanier Theological Library", "Library Service Center"}

def load_libraries(city, bbox, crs):
    f = CACHE / f"{city}_libraries.parquet"
    if f.exists():
        df = pd.read_parquet(f)
    else:
        g = OSM(str(pbf_path(city)), bounding_box=list(bbox)).get_pois(custom_filter={"amenity": ["library"]}).to_crs(crs)
        cen = g.geometry.centroid
        col = lambda k: g[k].fillna("").astype(str).values if k in g else np.full(len(g), "", dtype=object)
        df = pd.DataFrame({"x": cen.x.values, "y": cen.y.values, "name": col("name"), "operator": col("operator"),
                           "access": col("access"), "tags": col("tags")})
        df.to_parquet(f)
    text = df["name"] + " " + df["operator"] + " " + df["tags"]
    looks_public = (df["name"] + " " + df["operator"]).str.contains(PUBLIC_RE)
    bad_access = df["access"].isin(["private", "no", "customers", "permit", "students"])
    pub = (~text.str.contains(NONPUBLIC_RE) | looks_public) & ~bad_access
    pub = (pub & ~df["name"].isin(LIB_DROP)) | df["name"].isin(LIB_KEEP)
    return df.assign(public=pub.values)

# Housing-type features from OSM building tags. Coverage varies a lot (many buildings are just building=yes),
# so shares are shrunk toward the city average when few residential buildings are tagged nearby.
SMALL_HOME = {"house", "detached", "semidetached_house", "terrace", "townhouse", "bungalow", "duplex"}
LARGE_RES_M2, LARGE_LEVELS = 800, 3   # building=residential counts as multifamily if footprint or height is this big
SHRINK_M2 = 20000.0                   # footprint of "pseudo-evidence" at the city average

def load_buildings(city, bbox, crs):
    f = CACHE / f"{city}_buildings.parquet"
    if f.exists():
        return pd.read_parquet(f)
    g = OSM(str(pbf_path(city)), bounding_box=list(bbox)).get_buildings().to_crs(crs)
    g = g[g.geometry.geom_type.isin(["Polygon", "MultiPolygon"])]
    cen = g.geometry.centroid
    lv = pd.to_numeric(g["building:levels"], errors="coerce").values if "building:levels" in g else np.full(len(g), np.nan)
    df = pd.DataFrame({"x": cen.x.values.astype("float32"), "y": cen.y.values.astype("float32"),
                       "area": g.geometry.area.values.astype("float32"), "levels": lv.astype("float32"),
                       "building": g["building"].astype(str).astype("category").values})
    df.to_parquet(f)
    return df

def classify(b):
    t = b["building"].astype(str)
    mf = (t == "apartments") | ((t == "residential") & ((b["area"] >= LARGE_RES_M2) | (b["levels"] >= LARGE_LEVELS)))
    small = t.isin(SMALL_HOME)
    oth = (t == "residential") & ~mf
    return mf.values, small.values, oth.values, (t == "yes").values

def tree(xy): return cKDTree(xy) if len(xy) else None

# "Commercial activity" = places that bring foot/car traffic and noise: shops, food, drink, entertainment.
# Parking, benches, offices, clinics, schools, parks etc. are deliberately not counted.
COMMERCIAL_AMENITY = {"restaurant", "cafe", "bar", "pub", "fast_food", "food_court", "ice_cream", "biergarten",
                      "nightclub", "cinema", "theatre", "marketplace", "bank", "pharmacy", "car_wash", "fuel"}
def is_commercial(t):
    k, _, v = t.partition("=")
    return (k == "shop" and v not in ("vacant", "yes")) or (k == "amenity" and v in COMMERCIAL_AMENITY) \
        or (k == "craft") or (k == "leisure" and v in ("fitness_centre", "bowling_alley", "amusement_arcade"))

D = {}
for c in CITIES:
    poly = gpd.read_file(boundary_file(c)).geometry.iloc[0]
    crs = city_crs(poly.bounds)
    poi = pd.read_parquet(latest(f"{c}_pois_pbf_*.parquet"))
    roads, mpts = load_roads(c, poly.bounds, crs)
    sf = load_street_form(c, poly.bounds, crs)
    sn, se = sf[sf["kind"] == "node"], sf[sf["kind"] == "edge"]
    parks = load_parks(c, poly.bounds, crs)
    trails = load_trails(c, poly.bounds, crs, parks[parks["area_ha"] >= BIG_PARK_HA])
    libs = load_libraries(c, poly.bounds, crs)
    bld = load_buildings(c, poly.bounds, crs)
    mf, small, oth, yes = classify(bld)
    ar = bld["area"].values.astype("float64")
    res_tot = (ar * (mf | small | oth)).sum()
    bld_pack = dict(bldtree=tree(bld[["x", "y"]].values), a_mf=ar * mf, a_small=ar * small, a_oth=ar * oth,
                    a_yes=ar * yes, a_all=ar, prior_mf=(ar * mf).sum() / max(res_tot, 1),
                    prior_small=(ar * small).sum() / max(res_tot, 1))
    acs_pack = {}
    if USE_ACS:
        bgs = load_acs(c, poly, crs)
        tot = bgs[ACS_NUM].sum()
        acs_pack = dict(acs_geom=bgs.geometry.values, acs_tree=STRtree(bgs.geometry.values),
                        acs_area=np.maximum(bgs.geometry.area.values, 1.0),
                        acs_v={k: bgs[k].values.astype(float) for k in ACS_NUM},
                        acs_rent=bgs["rent_med"].values.astype(float), acs_val=bgs["val_med"].values.astype(float),
                        acs_prior={f: tot[nu] / max(tot[de], 1.0) for f, (nu, de) in SHARES.items()})
        print(c, "| census block groups", len(bgs), "| occupied units", int(tot["occ"]))
    big = parks[parks["area_ha"] >= BIG_PARK_HA]
    def pts(tags, p=poi):
        q = p[p["tag"].isin(tags)]; return tree(q[["x", "y"]].values)
    D[c] = dict(
        poly=poly, crs=crs,
        poi_all=tree(poi[["x", "y"]].values),
        library=tree(libs.loc[libs["public"], ["x", "y"]].values), market=pts(["shop=supermarket"]), hospital=pts(["amenity=hospital"]),
        school=pts(["amenity=school"]), eat=pts(["amenity=restaurant", "amenity=cafe", "amenity=fast_food"]),
        comm=tree(poi.loc[poi["tag"].map(is_commercial), ["x", "y"]].values),
        s_ntree=cKDTree(sn[["x", "y"]].values), s_ndeg=sn["deg"].values,
        s_etree=cKDTree(se[["x", "y"]].values), s_elen=se["len"].values, s_elocal=se["local"].values.astype(bool),
        rtree=cKDTree(roads[["x", "y"]].values), rlen=roads["len"].values, rmaj=roads["major"].values,
        majtree=tree(mpts[["x", "y"]].values), fwtree=tree(mpts.loc[mpts["freeway"], ["x", "y"]].values),
        pgeom=parks.geometry.values, ptree=STRtree(parks.geometry.values),
        btree=STRtree(big.geometry.values) if len(big) else None,
        trtree=tree(trails[["x", "y"]].values), libs_df=libs, **bld_pack, **acs_pack)
    print(c, "| POIs", len(poi), "| roads", len(roads), "| parks", len(parks), f"({len(big)} >= {BIG_PARK_HA} ha)", "| trail pts", len(trails),
          "| public libraries", int(libs["public"].sum()), "of", len(libs),
          "| buildings", len(bld), f"({yes.mean():.0%} tagged only 'yes', {(mf | small | oth).mean():.0%} residential-tagged)")

### Check the library filter. Fix mistakes by adding exact names to `LIB_DROP` / `LIB_KEEP` (cell above), then rerun from that cell.

In [ ]:
for c in CITIES:
    l = D[c]["libs_df"]
    nm = lambda m: sorted(l.loc[m, "name"].replace("", "(unnamed)"))
    print(f"== {c}: kept {int(l['public'].sum())}, dropped {int((~l['public']).sum())}")
    print("DROPPED:", nm(~l["public"]))
    print("KEPT:   ", nm(l["public"]))

## 2. Features per point (works for labeled places and for grid cells)

In [ ]:
GROUPS = {
    "quiet":    ["major_road_share", "dist_major", "dist_freeway"],
    "green":    ["park_frac", "dist_park", "dist_bigpark", "dist_trail", "trail_km_1500"],
    "vicinity": ["dist_library", "n_library_3km", "n_market_3km", "n_hospital_5km", "n_school_1500", "n_eat_1500"],
    "form":     ["intersection_density", "mean_edge_len"],
    "seclusion": ["local_street_share", "dead_end_density"],   # residential-only streets and cul-de-sacs = a self-contained pocket
    "activity": ["log_comm_400", "log_comm_800"],   # commercial activity right here (shops, restaurants, bars) - low = "nothing around"
    "center":   ["dist_downtown_km"],
}
if USE_ACS:   # census: what kind of housing, and who lives there
    GROUPS["housing"] = ["rental_complex_share", "condo_share", "attached_share", "detached_share"]
    GROUPS["family"] = ["kids_share", "senior_share", "owner_share"]
else:         # OSM-only guess (building tags are thin for condos)
    GROUPS["housing"] = ["mf_share", "small_home_share"]
COMPLEX_COL = "rental_complex_share" if USE_ACS else "mf_share"
FEATS = [f for cols in GROUPS.values() for f in cols]
# Tune these per anchor. Walkability-type "form" features get low weight on purpose.
WEIGHTS = {"quiet": 1.0, "green": 1.0, "vicinity": 1.0, "activity": 1.0, "seclusion": 0.75, "form": 0.5, "center": 0.5,
           "housing": 0.75, "family": 0.75}
# Hard filters on search results (None = off). These are NOT similarity features.
MAX_MF_SHARE = None       # drop cells where apartment complexes (renter-occupied, 5+ units; OSM guess if USE_ACS is False) exceed this share, e.g. 0.4
MAX_RENT = None           # drop cells whose median gross rent (US$/month) is above this, e.g. 1500
MAX_HOME_VALUE = None     # drop cells whose median home value (US$) is above this
W = np.array([WEIGHTS[g] / np.sqrt(len(cols)) for g, cols in GROUPS.items() for _ in cols])
LOGCOLS = [f for f in FEATS if f.startswith(("dist_", "n_")) or f in ("intersection_density", "mean_edge_len", "trail_km_1500", "dead_end_density")]

def to_xy(city, lat, lon):
    return Transformer.from_crs("EPSG:4326", D[city]["crs"], always_xy=True).transform(lon, lat)

def to_latlon(city, x, y):
    lon, lat = Transformer.from_crs(D[city]["crs"], "EPSG:4326", always_xy=True).transform(x, y)
    return lat, lon

def near(t, pts, cap):
    if t is None: return np.full(len(pts), float(cap))
    return np.minimum(t.query(np.c_[shapely.get_x(pts), shapely.get_y(pts)])[0], cap)

def pdist(t, pts, cap):
    if t is None: return np.full(len(pts), float(cap))
    _, dist = t.query_nearest(pts, return_distance=True, all_matches=False)
    return np.minimum(dist, cap)

def count(t, P, r):
    return np.zeros(len(P)) if t is None else t.query_ball_point(P, r, return_length=True).astype(float)

def bldg_stats(c, P, chunk=500):
    # footprint area (m²) of each building class within RADIUS_M; chunked because there are ~1M buildings
    d = D[c]; out = np.zeros((len(P), 5))
    for s in range(0, len(P), chunk):
        for k, idx in enumerate(d["bldtree"].query_ball_point(P[s:s + chunk], RADIUS_M)):
            if idx:
                out[s + k] = [d["a_mf"][idx].sum(), d["a_small"][idx].sum(), d["a_oth"][idx].sum(),
                              d["a_yes"][idx].sum(), d["a_all"][idx].sum()]
    return pd.DataFrame(out, columns=["mf", "small", "other_res", "untagged", "all"])

def housing_feats(c, S):
    d = D[c]; res = S["mf"] + S["small"] + S["other_res"]
    return pd.DataFrame({"mf_share": (S["mf"] + SHRINK_M2 * d["prior_mf"]) / (res + SHRINK_M2),
                         "small_home_share": (S["small"] + SHRINK_M2 * d["prior_small"]) / (res + SHRINK_M2)})

def acs_feats(c, bufs):
    """Census block groups -> per-cell housing/family shares (area-weighted over each 800 m circle, shrunk toward the city total)
    and median rent / home value (area x household weighted average of the block-group medians; NaN if no data nearby)."""
    d = D[c]; n = len(bufs); K = ACS_K
    S = {k: np.zeros(n) for k in ACS_NUM}
    rent, val = np.full(n, np.nan), np.full(n, np.nan)
    for i, b in enumerate(bufs):
        ids = d["acs_tree"].query(b)
        if len(ids) == 0: continue
        w = shapely.area(shapely.intersection(d["acs_geom"][ids], b)) / d["acs_area"][ids]   # share of each block group inside the circle
        for k in ACS_NUM: S[k][i] = (w * d["acs_v"][k][ids]).sum()
        for arr, key, out in ((d["acs_rent"], "renter", rent), (d["acs_val"], "owner", val)):
            m = ~np.isnan(arr[ids]); ww = (w * (d["acs_v"][key][ids] + 1.0))[m]
            if ww.sum() > 0: out[i] = (ww * arr[ids][m]).sum() / ww.sum()
    feats = {f: (S[nu] + K * d["acs_prior"][f]) / (S[de] + K) for f, (nu, de) in SHARES.items()}
    return feats, pd.DataFrame({"rent_median": rent, "home_value_median": val})

def compute(c, P, with_extra=False):
    d = D[c]; n = len(P); area_km2 = np.pi * (RADIUS_M / 1000) ** 2
    pts = shapely.points(P[:, 0], P[:, 1])
    # quiet: share of nearby road length that is major, distance to major/freeway
    tot, maj = np.zeros(n), np.zeros(n)
    for k, idx in enumerate(d["rtree"].query_ball_point(P, RADIUS_M)):
        if idx:
            l = d["rlen"][idx]; tot[k] = l.sum(); maj[k] = l[d["rmaj"][idx]].sum()
    # green: park area share within 800 m
    bufs = shapely.buffer(pts, RADIUS_M); pf = np.zeros(n)
    for k, b in enumerate(bufs):
        ids = d["ptree"].query(b)
        if len(ids): pf[k] = shapely.area(shapely.intersection(d["pgeom"][ids], b)).sum() / b.area
    # street form (driving network): intersections and dead ends per km2, mean block length, share of local streets
    inter, dead, mel, loc = np.zeros(n), np.zeros(n), np.zeros(n), np.zeros(n)
    for k, idx in enumerate(d["s_ntree"].query_ball_point(P, RADIUS_M)):
        if idx:
            dg = d["s_ndeg"][idx]; inter[k] = (dg >= 3).sum() / area_km2; dead[k] = (dg == 1).sum() / area_km2
    for k, idx in enumerate(d["s_etree"].query_ball_point(P, RADIUS_M)):
        if idx:
            l = d["s_elen"][idx]; mel[k] = l.mean(); loc[k] = l[d["s_elocal"][idx]].sum() / max(l.sum(), 1)
    # centrality
    tx = Transformer.from_crs("EPSG:4326", d["crs"], always_xy=True).transform(DOWNTOWN[c][1], DOWNTOWN[c][0])
    out = {
        "major_road_share": np.where(tot > 0, maj / np.maximum(tot, 1), 0.0),
        "dist_major": near(d["majtree"], pts, 5000), "dist_freeway": near(d["fwtree"], pts, 8000),
        "park_frac": pf, "dist_park": pdist(d["ptree"], pts, 3000), "dist_bigpark": pdist(d["btree"], pts, 8000),
        "dist_trail": near(d["trtree"], pts, 5000), "trail_km_1500": count(d["trtree"], P, 1500) * 50 / 1000,
        "dist_library": near(d["library"], pts, 8000), "n_library_3km": count(d["library"], P, 3000),
        "n_market_3km": count(d["market"], P, 3000), "n_hospital_5km": count(d["hospital"], P, 5000),
        "n_school_1500": count(d["school"], P, 1500), "n_eat_1500": count(d["eat"], P, 1500),
        "intersection_density": inter, "mean_edge_len": mel, "local_street_share": loc, "dead_end_density": dead,
        "log_comm_400": np.log1p(count(d["comm"], P, 400)),
        "log_comm_800": np.log1p(count(d["comm"], P, RADIUS_M)),
        "dist_downtown_km": np.hypot(P[:, 0] - tx[0], P[:, 1] - tx[1]) / 1000,
    }
    if USE_ACS:
        A_, X_ = acs_feats(c, bufs); out.update(A_)
    else:
        H = housing_feats(c, bldg_stats(c, P))
        out["mf_share"], out["small_home_share"] = H["mf_share"].values, H["small_home_share"].values
        X_ = pd.DataFrame({"rent_median": np.full(n, np.nan), "home_value_median": np.full(n, np.nan)})
    F = pd.DataFrame(out)[FEATS]
    return (F, X_) if with_extra else F

def make_grid(c):
    d = D[c]; w, s, e, n = d["poly"].bounds
    t = Transformer.from_crs("EPSG:4326", d["crs"], always_xy=True)
    (x0, y0), (x1, y1) = t.transform(w, s), t.transform(e, n)
    X, Y = np.meshgrid(np.arange(x0, x1, SPACING_M), np.arange(y0, y1, SPACING_M))
    poly_xy = gpd.GeoSeries([d["poly"]], crs=4326).to_crs(d["crs"]).iloc[0]
    m = shapely.contains_xy(poly_xy, X.ravel(), Y.ravel())
    return np.c_[X.ravel(), Y.ravel()][m]

GRID = {}
for c in CITIES:
    P = make_grid(c); F, X = compute(c, P, True)
    # usable = some urban signal of any kind (all POIs); this is NOT a feature any more, only the land mask
    ok = (count(D[c]["poi_all"], P, RADIUS_M) >= MIN_POI_800)
    GRID[c] = dict(P=P, F=F, X=X, ok=ok)
    print(c, "cells:", len(P), "usable:", int(GRID[c]["ok"].sum()))

## 3. Labeled places: raw values and percentile within their city (usable cells)

In [ ]:
rows, xrows, names, diag = [], [], [], []
for _, r in labels.iterrows():
    if r["city"] not in D: continue     # e.g. Richmond is outside the Houston boundary
    x, y = to_xy(r["city"], r["lat"], r["lon"])
    F_, X_ = compute(r["city"], np.array([[x, y]]), True)
    rows.append(F_.iloc[0]); xrows.append(X_.iloc[0]); names.append(r["name"])
    diag.append(bldg_stats(r["city"], np.array([[x, y]])).iloc[0])
L = pd.DataFrame(rows, index=names).astype(float)
LX = pd.DataFrame(xrows, index=names).astype(float)
pd.set_option("display.width", 220, "display.max_columns", 30)
print(L.round(2).T.to_string())
print("\nMedian rent (US$/month) and median home value (US$) near each place (census, filters only):")
print(LX.round(0).T.to_string())

# Coverage check for the housing features: how much of the nearby footprint is tagged residential vs just 'yes'?
Dg = pd.DataFrame(diag, index=names); res_m2 = Dg["mf"] + Dg["small"] + Dg["other_res"]
print("\nBuilding tag coverage within 800 m (low tagged share = housing features are shrunk toward the city average):")
print(pd.DataFrame({"mf_m2": Dg["mf"].round(), "small_home_m2": Dg["small"].round(), "other_res_m2": Dg["other_res"].round(),
                    "untagged_yes_m2": Dg["untagged"].round(),
                    "residential_tagged_share": (res_m2 / Dg["all"].clip(lower=1)).round(2)}).to_string())

print("\nPercentile of each place among usable cells of its city (0 = lowest, 1 = highest):")
pct = {}
for n_, r in labels.set_index("name").loc[L.index].iterrows():
    G = GRID[r["city"]]["F"][GRID[r["city"]]["ok"]]
    pct[n_] = [(G[f] < L.loc[n_, f]).mean() for f in FEATS]
print(pd.DataFrame(pct, index=FEATS).round(2).to_string())

**Look at this table before the search.** Do the liked places (home area, West U, Museum District) share a pattern on quiet / green / vicinity?
Do Summer St and Midtown differ (more major roads, higher intersection density)? If the labeled places do not separate on these features, searching will not help yet.

## 4. Search: nearest cells to a labeled anchor (one query per anchor, NMS, biggest gaps)

In [ ]:
def transform(F):
    T = F.copy()
    for c in LOGCOLS: T[c] = np.log1p(T[c])
    return T

ALL = pd.concat([GRID[c]["F"][GRID[c]["ok"]] for c in CITIES])
TA = transform(ALL); MU, SD = TA.mean(), TA.std() + 1e-9
STATS = {}
for c_ in CITIES:
    t_ = transform(GRID[c_]["F"][GRID[c_]["ok"]]); STATS[c_] = (t_.mean(), t_.std() + 1e-9)
# "pooled": one mean/sd over all cities, so absolute levels are comparable (200 m to a library means the same everywhere).
# "per_city": each city is centred on its own average, which removes city-wide differences (CLAUDE.md's leakage fix).
# "hybrid": per-city for the features listed in PER_CITY_FEATS, pooled for the rest (fill it from the table in 4b).
# Section 4b measured the leakage: pooled predicted the city at 0.74 (chance 0.25), per-city removed it, so per-city is the default.
ZMODE = "per_city"
PER_CITY_FEATS = set()

def Z(F, city, mode=None):
    mode = mode or ZMODE
    if mode == "pooled": mu, sd = MU, SD
    elif mode == "per_city": mu, sd = STATS[city]
    else:   # hybrid
        mu, sd = MU.copy(), SD.copy()
        for f in PER_CITY_FEATS: mu[f], sd[f] = STATS[city][0][f], STATS[city][1][f]
    return ((transform(F) - mu) / sd).clip(-3, 3).values

def search(anchor, target, k=8, zmode=None):
    a = labels[labels["name"] == anchor].iloc[0]
    za = Z(L.loc[[anchor]], a["city"], zmode)[0]
    T = GRID[target]; ZT = Z(T["F"], target, zmode)
    d = np.sqrt((((ZT - za) * W) ** 2).sum(1)); d[~T["ok"]] = 1e9
    if MAX_MF_SHARE is not None: d[T["F"][COMPLEX_COL].values > MAX_MF_SHARE] = 1e9
    if MAX_RENT is not None: d[T["X"]["rent_median"].values > MAX_RENT] = 1e9              # unknown rent (NaN) is kept
    if MAX_HOME_VALUE is not None: d[T["X"]["home_value_median"].values > MAX_HOME_VALUE] = 1e9
    if a["city"] == target:   # exclude the anchor's own neighborhood
        ax, ay = to_xy(a["city"], a["lat"], a["lon"])
        d[np.hypot(T["P"][:, 0] - ax, T["P"][:, 1] - ay) < NMS_M] = 1e9
    chosen = []
    for j in np.argsort(d):
        if d[j] >= 1e9: break
        if all(np.hypot(*(T["P"][j] - T["P"][c])) >= NMS_M for c in chosen): chosen.append(j)
        if len(chosen) == k: break
    out = []
    for j in chosen:
        lat, lon = to_latlon(target, *T["P"][j])
        gap = np.argsort(-np.abs((ZT[j] - za) * W))[:3]
        out.append(dict(dist=round(float(d[j]), 2), rent=round(float(T["X"]["rent_median"].values[j]), -1),
                        home_value=round(float(T["X"]["home_value_median"].values[j]), -3), lat=round(lat, 5), lon=round(lon, 5),
                        map=f"https://maps.google.com/?q={lat:.5f},{lon:.5f}",
                        biggest_gaps=", ".join(np.array(FEATS)[gap])))
    return pd.DataFrame(out)

pd.set_option("display.max_colwidth", 90)
for anchor in L.index:
    if labels.set_index("name").loc[anchor, "label"] not in ("like", "like_conditional"): continue
    for target in CITIES:
        print(f"\n=== {anchor}  ->  {target} (lower dist = closer) ===")
        print(search(anchor, target).to_string(index=False))

## 4b. Cross-city checks: does the city leak into the features, and does the z-scoring choice change the answer?

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score

def city_probe(mode):
    """Balanced accuracy of predicting the CITY from a cell's feature vector. Chance = 1 / n_cities.
    Cells overlap (800 m radius, 400 m spacing), so folds are spatial 5 km tiles, otherwise neighbours leak across folds."""
    X = np.vstack([Z(GRID[c]["F"][GRID[c]["ok"]], c, mode) for c in CITIES])
    y = np.concatenate([[c] * int(GRID[c]["ok"].sum()) for c in CITIES])
    tiles = np.concatenate([[f"{c}:{int(x // 5000)}:{int(yy // 5000)}" for x, yy in GRID[c]["P"][GRID[c]["ok"]]] for c in CITIES])
    g = pd.factorize(tiles)[0]
    return cross_val_score(LogisticRegression(max_iter=500, class_weight="balanced"), X, y, groups=g,
                           cv=GroupKFold(5), scoring="balanced_accuracy").mean()

if len(CITIES) > 1:
    print(f"chance = {1 / len(CITIES):.2f}  (higher = the city is easier to tell from the features = more leakage)")
    for mode in ("pooled", "per_city"):
        print(f"city predictable, {mode:9s} z-scoring: {city_probe(mode):.2f}")

def km(lat1, lon1, lat2, lon2):
    return np.hypot((lat1 - lat2) * 111.0, (lon1 - lon2) * 111.0 * np.cos(np.radians((lat1 + lat2) / 2)))

print("\nHow many of an anchor's top-8 results are the same place (within 1 km) under pooled vs per-city z-scoring?")
rows_ = []
for anchor in L.index:
    if labels.set_index("name").loc[anchor, "label"] not in ("like", "like_conditional"): continue
    for target in CITIES:
        A_, B_ = search(anchor, target, zmode="pooled"), search(anchor, target, zmode="per_city")
        same = sum(any(km(r.lat, r.lon, s.lat, s.lon) < 1.0 for s in B_.itertuples()) for r in A_.itertuples())
        rows_.append((anchor, target, f"{same}/{len(A_)}", round(A_["dist"].iloc[0], 2), round(B_["dist"].iloc[0], 2)))
print(pd.DataFrame(rows_, columns=["anchor", "target", "same_in_top8", "best_dist_pooled", "best_dist_per_city"]).to_string(index=False))
print("\nIf the probe is far above chance with pooled but near chance per-city, cities differ in overall level and per-city is the safer choice for cross-city matching."
      " (A per-city score BELOW chance is an artifact of centring each city and holding out whole tiles; read it as 'no city signal left'.)"
      " Set ZMODE above ('pooled', 'per_city' or 'hybrid') and rerun section 4 to see another list.")

### 4c. Which features carry the city signal?  (average pooled z-score of the usable cells per city)

In [ ]:
Zp = {c: pd.DataFrame(Z(GRID[c]["F"][GRID[c]["ok"]], c, "pooled"), columns=FEATS) for c in CITIES}
M = pd.DataFrame({c: Zp[c].mean() for c in CITIES})
M["spread"] = M[CITIES].max(axis=1) - M[CITIES].min(axis=1)
M["group"] = [next(g for g, cols in GROUPS.items() if f in cols) for f in M.index]
pd.set_option("display.width", 200)
print("Average pooled z per city. A feature with a big spread (> ~0.5) is one where the cities sit at different levels:")
print(M.sort_values("spread", ascending=False).round(2).to_string())
print("\nMean spread by group:")
print(M.groupby("group")["spread"].mean().sort_values(ascending=False).round(2).to_string())
print("\nTo go hybrid: set ZMODE = 'hybrid' and PER_CITY_FEATS = {...features to centre per city...} in section 4, then rerun it."
      " Centre per city the features whose gap is an artifact (mapping completeness, how the search area was drawn, e.g. dist_downtown_km);"
      " leave pooled the ones where a real difference should count (e.g. how quiet or green a place is in absolute terms).")

## 5. Not built yet (next steps, in order)
1. **Safety**: crime counts as a within-city percentile plus a hard filter (sources and definitions differ per city, so not a cross-city feature).
2. **Tree canopy** (NLCD/USFS raster) for "green" beyond park polygons.
3. **Negative anchors**: use `dislike` rows (needs the previous apartment's location, and a grid that covers Richmond) to push down cells that look more like a disliked place than a liked one.
4. Judge the cross-city lists (Atlanta / Austin / Seattle) by visiting or by map; the leakage probe in 4b only checks for gross city effects.
5. Per-anchor weights (`WEIGHTS`) from the labeled places, as in the CLAUDE.md few-shot plan.

## 6. Export the cache (old + new extracts) as a clean folder for a new dataset version

In [ ]:
EXPORT = Path("/kaggle/working/export") if os.path.exists("/kaggle") else CACHE.parent / "export"
EXPORT.mkdir(parents=True, exist_ok=True)
files = [p for p in CACHE.iterdir() if p.is_file() and p.suffix in (".parquet", ".geojson")]
for p in files: shutil.copy2(p, EXPORT / p.name)
print(f"{len(files)} files, {sum(p.stat().st_size for p in files) / 1e6:.0f} MB in {EXPORT}")
for c in CITIES:
    for n in (f"{c}_roads_class", f"{c}_major_pts", f"{c}_parks", f"{c}_trail_ways", f"{c}_libraries", f"{c}_buildings", f"{c}_streets_drive") + ((f"{c}_acs_bg_{ACS_YEAR}",) if USE_ACS else ()):
        print("ok      " if (EXPORT / f"{n}.parquet").exists() else "MISSING ", n)
if os.path.exists("/kaggle"):   # one file to download from the Output panel instead of 15+
    z = shutil.make_archive("/kaggle/working/cache_export", "zip", str(EXPORT))
    print(f"zip: {z} ({os.path.getsize(z) / 1e6:.0f} MB) - download it from Data -> Output -> /kaggle/working, unzip on your laptop, upload the files as a new dataset version.")
print("Next: Data -> Output -> export folder -> update the cache dataset (New Version).")